**PS4 — Hourly Anomaly Detection**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture  : CUBIC MARS AWS v6
Target        : ensemble_anomaly_flag (binary: 1 if 2+ of 3 active signals)
Grain         : one row per DEVICE_ID per hour
Signals       : S1=SPC 3-sigma  S2=Isolation Forest  S3=Tap rejection >5%
                S4=SolarWinds network [INACTIVE — telemetry not available in Gold]
                S5=Bus ops ratio [REMOVED 2026-06-12 — CTA_REAL_TIME_BUS_DATA dropped]
NOTE          : signal_bus_ops (S5) and signal_network (S4) always default to 0.
                Effective ensemble = 2-of-3 active signals (S1, S2, S3).
Models        : 3-signal ensemble voting + HDBSCAN clustering + supervised XGBoost
Feasibility   : 95%  (5% gap: bus device IP coverage incomplete in SolarWinds)

---

In [ ]:
# ── CELL 1 : Install ──────────────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs","pyarrow",
    "scikit-learn>=1.4","xgboost>=2.0","lightgbm>=4.3",
    "hdbscan>=0.8","optuna>=3.6","shap>=0.45","mlflow>=2.13", "sagemaker-mlflow",
    "boto3","sagemaker>=2.220,<3.0","pandas>=2.0","matplotlib","seaborn","scipy",
]
subprocess.check_call([sys.executable,"-m","pip","install","-q","--upgrade"]+pkgs)
print("OK")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile
from scipy import stats

# deltalake removed: gold tables exported as Parquet; pd.read_parquet() used in CELL 4
from sklearn.model_selection         import TimeSeriesSplit
from sklearn.preprocessing           import StandardScaler
from sklearn.ensemble                import (IsolationForest,
                                             RandomForestClassifier,
                                             StackingClassifier)
from sklearn.linear_model            import LogisticRegression
from sklearn.pipeline                import Pipeline
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.metrics                 import (roc_auc_score, f1_score,
                                             average_precision_score,
                                             confusion_matrix,
                                             classification_report)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
import hdbscan

import mlflow, mlflow.sklearn
from mlflow.models.signature import infer_signature
try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session                       # sagemaker >= 3
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role            # sagemaker >= 3
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
ARTIFACT_BUCKET     = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET         = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX         = "chicago/gold/device_ps4_hourly"
GOLD_S3             = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION          = Session()
ROLE                = get_execution_role()
REGION              = boto3.Session().region_name
SM_CLIENT           = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT           = boto3.client("s3", region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)  # sagemaker-mlflow resolves arn: scheme

EXPERIMENT_NAME     = "chicago-ps4-anomaly-detection"
MLFLOW_MODEL_NAME   = "ps4-anomaly-detection"
MODEL_REGISTRY_NAME = "chicago-ps4-anomaly-detection"
FS_GROUP_NAME       = "chicago-ps4-features"
ENDPOINT_NAME       = "chicago-ps4-anomaly-v1"
TARGET              = "ensemble_anomaly_flag"
SIGNAL_THRESHOLD    = 2   # 2-of-3 active signals (S4 network + S5 bus_ops inactive)

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3: {GOLD_S3}")

In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────
print(f"Loading Gold table from: {GOLD_S3}")
df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

df["hour_dt"] = pd.to_datetime(df["hour_dt"])

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['hour_dt'].min()}  →  {df['hour_dt'].max()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"Anomaly rate  : {df[TARGET].mean():.2%}")
print(f"\nData source: {DATA_SOURCE}  |  Gold version: {GOLD_VERSION}")

In [ ]:
# ── CELL 5 : Build signal ensemble flags ─────────────────────────────────────
# 3 active signals: S1=SPC, S2=IsolationForest, S3=Tap rejection
# S4 (SolarWinds network) and S5 (Bus ops) are permanently inactive — see header.

# Signal 1: SPC — event rate 3-sigma control chart
# Flag = 1 when |event_rate - rolling_mean| > 3 * rolling_std
if "daily_event_count" in df.columns:
    df_s = df.sort_values(["DEVICE_ID","hour_dt"])
    roll_mean = (df_s.groupby("DEVICE_ID")["daily_event_count"]
                     .transform(lambda x: x.rolling(24,min_periods=6).mean()))
    roll_std  = (df_s.groupby("DEVICE_ID")["daily_event_count"]
                     .transform(lambda x: x.rolling(24,min_periods=6).std().fillna(1)))
    df["signal_spc"] = (
        (df["daily_event_count"] - roll_mean).abs() > 3*roll_std).astype(int)
elif "signal_spc" not in df.columns:
    df["signal_spc"] = 0

# Signal 2: Isolation Forest score (derived during model training — cell 11)
# Placeholder column — filled in Cell 11
if "signal_isolation_forest" not in df.columns:
    df["signal_isolation_forest"] = 0

# Signal 3: Tap rejection rate > 5%
if "tap_rejection_rate_pct" in df.columns:
    df["signal_tap_rejection"] = (df["tap_rejection_rate_pct"] > 5.0).astype(int)
elif "signal_tap_rejection" not in df.columns:
    df["signal_tap_rejection"] = 0

# Signal 4: SolarWinds network — INACTIVE (telemetry not available in Gold)
if "signal_network" not in df.columns:
    df["signal_network"] = 0

# Signal 5: Bus ops ratio — REMOVED 2026-06-12
# CTA_REAL_TIME_BUS_DATA / bus_realtime_daily REMOVED 2026-06-12
# bus_in_service_ratio is no longer available in Gold; signal permanently 0.
if "signal_bus_ops" not in df.columns:
    df["signal_bus_ops"] = 0

# Signal 6: Revenue zero — device had transactions but collected zero fare (S21 use_revenue_daily)
# Gold column: use_revenue_zero_flag (1 when daily_txn_count > 0 AND daily_fare_due_cents = 0)
# Active signal (Signal 4=SolarWinds INACTIVE, Signal 5=BusOps REMOVED)
# Covers fare system failures invisible to event/metric/tap signals
if "use_revenue_zero_flag" in df.columns:
    df["signal_revenue_zero"] = df["use_revenue_zero_flag"].fillna(0).astype(int)
elif "signal_revenue_zero" not in df.columns:
    df["signal_revenue_zero"] = 0

SIGNAL_COLS = ["signal_spc","signal_isolation_forest","signal_tap_rejection",
               "signal_network","signal_bus_ops","signal_revenue_zero"]
SIGNAL_COLS = [c for c in SIGNAL_COLS if c in df.columns]

# Derive ensemble_anomaly_flag if not in Gold
df["signal_active_count"] = df[SIGNAL_COLS].sum(axis=1)
if TARGET not in df.columns:
    df[TARGET] = (df["signal_active_count"] >= SIGNAL_THRESHOLD).astype(int)

print(f"\nSignal active rates:")
for s in SIGNAL_COLS:
    print(f"  {s}: {df[s].mean():.2%}")
print(f"\nEnsemble anomaly rate: {df[TARGET].mean():.2%}")
print(f"(2+ of 3 active signals: signal_spc, signal_isolation_forest, signal_tap_rejection)")

In [ ]:
# ── CELL 6 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Signal co-activation heatmap
if len(SIGNAL_COLS) >= 2:
    sig_corr = df[SIGNAL_COLS].corr()
    sns.heatmap(sig_corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                ax=axes[0,0], linewidths=0.5)
    axes[0,0].set_title("Signal Co-activation Correlation")

# Anomaly flag distribution
df[TARGET].value_counts().plot(kind="bar", ax=axes[0,1],
    color=["#43A047","#E53935"], edgecolor="white")
axes[0,1].set_title("ensemble_anomaly_flag distribution")

# Hourly anomaly pattern
df.groupby(df["hour_dt"].dt.hour)[TARGET].mean().plot(
    kind="bar", ax=axes[1,0], color="#1565C0")
axes[1,0].set_title("Anomaly rate by hour of day")

# By device type
if "device_type" in df.columns:
    df.groupby("device_type")[TARGET].mean().plot(
        kind="bar", ax=axes[1,1], color="#7B1FA2")
    axes[1,1].set_title("Anomaly rate by device type")

plt.tight_layout(); plt.savefig("/tmp/ps4_eda.png", dpi=100); plt.show()

In [ ]:
# ── CELL 7 : Feature engineering & splits ─────────────────────────────────────
ID_COLS = ["DEVICE_ID","hour_dt","split"] + SIGNAL_COLS + ["signal_active_count"]
ID_COLS = [c for c in ID_COLS if c in df.columns]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET]
    and pd.api.types.is_numeric_dtype(df[c])
]
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())
print(f"Features: {len(FEATURE_COLS)}")

train_df = df[df["hour_dt"] <  "2025-10-01"]
val_df   = df[(df["hour_dt"] >= "2025-10-01") & (df["hour_dt"] < "2026-01-01")]
test_df  = df[df["hour_dt"] >= "2026-01-01"]

X_train, y_train = train_df[FEATURE_COLS].values, train_df[TARGET].values
X_val,   y_val   = val_df[FEATURE_COLS].values,   val_df[TARGET].values
X_test,  y_test  = test_df[FEATURE_COLS].values,  test_df[TARGET].values
X_cv             = np.vstack([X_train,X_val])
y_cv             = np.concatenate([y_train,y_val])

neg, pos         = (y_train==0).sum(), (y_train==1).sum()
SCALE_POS_WEIGHT = round(neg/pos, 2)
print(f"Train:{len(X_train):,}  Val:{len(X_val):,}  Test:{len(X_test):,}")
print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")

In [ ]:
# ── CELL 8 : Feature Store ────────────────────────────────────────────────────
fs_df = train_df[FEATURE_COLS+[TARGET,"DEVICE_ID","hour_dt"]].copy()
fs_df["event_time"] = fs_df["hour_dt"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
fs_df               = fs_df.drop(columns=["hour_dt"])

def dtype_to_fs(dtype):
    if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
    if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
    return FeatureTypeEnum.STRING

feature_group=FeatureGroup(name=FS_GROUP_NAME,sagemaker_session=SM_SESSION,
    feature_definitions=[
        FeatureDefinition(feature_name=c,feature_type=dtype_to_fs(fs_df[c].dtype))
        for c in fs_df.columns])
try:
    feature_group.create(
        s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
        record_identifier_name="DEVICE_ID",event_time_feature_name="event_time",
        role_arn=ROLE,enable_online_store=True,
        description="PS4 hourly anomaly features — Chicago Ventra")
    status=""
    while status!="Created":
        time.sleep(15)
        status=feature_group.describe().get("FeatureGroupStatus")
        print(f"  {status}")
except Exception as e: print(f"Feature Group: {e}")
feature_group.ingest(data_frame=fs_df.reset_index(drop=True),max_workers=4,wait=True)
print(f"Ingested {len(fs_df):,} → {FS_GROUP_NAME}")

In [ ]:
# ── CELL 9 : Helpers ──────────────────────────────────────────────────────────
COMMON_TAGS={"ps":"PS4","target":"ensemble_anomaly_flag","city":"chicago",
             "gold_version":str(GOLD_VERSION),"architecture":"CUBIC-MARS-AWS-v6"}

def evaluate(model,X,y,split="test"):
    prob=model.predict_proba(X)[:,1]; pred=(prob>=0.5).astype(int)
    return {f"{split}_auc" :round(roc_auc_score(y,prob),4),
            f"{split}_ap"  :round(average_precision_score(y,prob),4),
            f"{split}_f1"  :round(f1_score(y,pred),4)}

def log_cm(model,X,y,name):
    pred=(model.predict_proba(X)[:,1]>=0.5).astype(int)
    cm=confusion_matrix(y,pred)
    fig,ax=plt.subplots(figsize=(4,3))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,
                xticklabels=["Normal","Anomaly"],
                yticklabels=["Normal","Anomaly"])
    ax.set_title(name); fig.tight_layout()
    fig.savefig(f"/tmp/cm_{name}.png",dpi=80); plt.close(fig)
    mlflow.log_artifact(f"/tmp/cm_{name}.png")

In [ ]:
# ── CELL 10 : Signal 1 — SPC 3-sigma control chart (logged as analysis) ───────
# Compute SPC metrics on training set. Log UCL/LCL thresholds to MLflow.

spc_results={}
if "daily_event_count" in df.columns:
    for device_type in (df["device_type"].unique() if "device_type" in df.columns
                        else ["ALL"]):
        if device_type=="ALL":
            subset=train_df["daily_event_count"].dropna()
        else:
            subset=train_df[train_df["device_type"]==device_type]["daily_event_count"].dropna()
        if len(subset)<50: continue
        mean,std=subset.mean(),subset.std()
        spc_results[device_type]={"mean":round(mean,4),
                                   "ucl":round(mean+3*std,4),
                                   "lcl":max(0,round(mean-3*std,4))}

with mlflow.start_run(run_name="spc_signal1_analysis") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"spc_control_chart","signal":"S1"})
    for dt_name,vals in spc_results.items():
        mlflow.log_metric(f"{dt_name}_mean",vals["mean"])
        mlflow.log_metric(f"{dt_name}_ucl", vals["ucl"])
        mlflow.log_metric(f"{dt_name}_lcl", vals["lcl"])
    with open("/tmp/spc_thresholds.json","w") as f: json.dump(spc_results,f,indent=2)
    mlflow.log_artifact("/tmp/spc_thresholds.json")
    mlflow.log_param("sigma_threshold",3)
    mlflow.log_param("rolling_window_hours",24)
    SPC_RUN=run.info.run_id

print(f"SPC thresholds logged for {len(spc_results)} device types")

In [ ]:
# ── CELL 11 : Signal 2 — Isolation Forest ─────────────────────────────────────
# Unsupervised anomaly detection on raw sensor/metric features.
# contamination = expected anomaly rate in training data.

IF_FEATURES = [c for c in FEATURE_COLS
               if any(k in c.lower() for k in
                      ["metric","response","percent_loss","kpi","rate","count"])]
IF_FEATURES = IF_FEATURES if IF_FEATURES else FEATURE_COLS[:15]

scaler  = StandardScaler()
X_if_tr = scaler.fit_transform(train_df[IF_FEATURES].values)
X_if_te = scaler.transform(test_df[IF_FEATURES].values)

with mlflow.start_run(run_name="isolation_forest_signal2") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"isolation_forest","signal":"S2"})

    iforest = IsolationForest(
        n_estimators       = 200,
        contamination      = float(y_train.mean()),   # use known anomaly rate
        max_samples        = "auto",
        random_state       = 42,
        n_jobs             = -1,
    )
    iforest.fit(X_if_tr)

    # Scores: lower = more anomalous
    scores_train = iforest.decision_function(X_if_tr)
    scores_test  = iforest.decision_function(X_if_te)
    labels_test  = (iforest.predict(X_if_te) == -1).astype(int)   # -1=anomaly→1

    mlflow.log_params({"n_estimators":200,"contamination":round(y_train.mean(),4),
                       "if_features":len(IF_FEATURES)})
    mlflow.log_metric("if_anomaly_rate_test", round(labels_test.mean(),4))

    # How well does IF predict ensemble_anomaly_flag?
    if len(labels_test)==len(y_test):
        if_f1=f1_score(y_test,labels_test)
        mlflow.log_metric("if_f1_vs_ensemble",round(if_f1,4))
        print(f"Isolation Forest F1 vs ensemble_anomaly_flag: {if_f1:.4f}")

    joblib.dump(iforest,"/tmp/ps4_iforest.joblib")
    joblib.dump(scaler, "/tmp/ps4_if_scaler.joblib")
    mlflow.log_artifact("/tmp/ps4_iforest.joblib","isolation_forest")
    mlflow.log_artifact("/tmp/ps4_if_scaler.joblib","isolation_forest")
    IF_RUN=run.info.run_id

    # Add IF score as a feature for supervised model
    df_if_score=pd.concat([
        pd.Series(iforest.decision_function(scaler.transform(
            train_df[IF_FEATURES].values)),index=train_df.index,name="if_score"),
        pd.Series(iforest.decision_function(scaler.transform(
            val_df[IF_FEATURES].values)),index=val_df.index,name="if_score"),
        pd.Series(iforest.decision_function(scaler.transform(
            test_df[IF_FEATURES].values)),index=test_df.index,name="if_score"),
    ])
    df=df.join(df_if_score)
    if "if_score" not in FEATURE_COLS:
        FEATURE_COLS.append("if_score")
    # Rebuild arrays with new feature
    X_train=train_df.join(df_if_score)[FEATURE_COLS].fillna(0).values
    X_val  =val_df.join(df_if_score)[FEATURE_COLS].fillna(0).values
    X_test =test_df.join(df_if_score)[FEATURE_COLS].fillna(0).values
    X_cv   =np.vstack([X_train,X_val])

print(f"IF score added as feature. Feature count now: {len(FEATURE_COLS)}")

In [ ]:
# ── CELL 12 : HDBSCAN — anomaly cluster discovery ────────────────────────────
# HDBSCAN over DBSCAN — handles varying density, no eps parameter needed.
# Noise points (label=-1) are candidate anomalies.
# Used for: identifying anomaly archetypes / cluster-based features.

with mlflow.start_run(run_name="hdbscan_clustering") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"hdbscan_clustering","signal":"derived"})

    # Use normalised features (subset for speed)
    X_cluster = scaler.transform(train_df[IF_FEATURES].fillna(0).values)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size  = max(50, int(len(X_cluster)*0.001)),
        min_samples       = 10,
        cluster_selection_epsilon = 0.1,
        metric            = "euclidean",
        core_dist_n_jobs  = -1,
    )
    cluster_labels = clusterer.fit_predict(X_cluster)

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    noise_rate  = (cluster_labels==-1).mean()

    mlflow.log_params({"min_cluster_size":clusterer.min_cluster_size,
                       "min_samples":clusterer.min_samples,
                       "metric":"euclidean"})
    mlflow.log_metrics({"n_clusters":n_clusters,"noise_rate":round(noise_rate,4)})

    # Visualise cluster sizes
    cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
    fig,ax=plt.subplots(figsize=(10,4))
    cluster_sizes.iloc[:20].plot(kind="bar",ax=ax,color="#1565C0",edgecolor="white")
    ax.set_title(f"HDBSCAN — {n_clusters} clusters (noise rate={noise_rate:.1%})")
    ax.set_xlabel("Cluster ID (-1=noise)")
    fig.tight_layout()
    fig.savefig("/tmp/hdbscan_clusters.png",dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/hdbscan_clusters.png")
    HDBSCAN_RUN=run.info.run_id

    print(f"HDBSCAN: {n_clusters} clusters  noise_rate={noise_rate:.1%}")

In [ ]:
# ── CELL 13 : MODEL 1 — XGBoost supervised (on ensemble features) ─────────────
XGB_PARAMS=dict(
    n_estimators=600,max_depth=6,learning_rate=0.05,
    subsample=0.8,colsample_bytree=0.8,min_child_weight=10,
    scale_pos_weight=SCALE_POS_WEIGHT,
    eval_metric="auc",early_stopping_rounds=40,
    tree_method="hist",random_state=42,n_jobs=-1)

with mlflow.start_run(run_name="xgboost_anomaly") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"xgboost_supervised"})
    xgb=XGBClassifier(**XGB_PARAMS)
    xgb.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
    vm=evaluate(xgb,X_val,y_val,"val"); tm=evaluate(xgb,X_test,y_test,"test")
    mlflow.log_params({**XGB_PARAMS,"best_iter":xgb.best_iteration})
    mlflow.log_metrics({**vm,**tm}); log_cm(xgb,X_test,y_test,"xgb")
    sig=infer_signature(X_train,xgb.predict_proba(X_train))
    mlflow.sklearn.log_model(xgb,"model",signature=sig); XGB_RUN=run.info.run_id
print(f"XGB anomaly  val_auc={vm['val_auc']}  test_auc={tm['test_auc']}")

In [ ]:
# ── CELL 14 : MODEL 2 — LightGBM ─────────────────────────────────────────────
with mlflow.start_run(run_name="lightgbm_anomaly") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"lightgbm_supervised"})
    lgb=LGBMClassifier(n_estimators=600,num_leaves=63,learning_rate=0.05,
                        subsample=0.8,colsample_bytree=0.8,
                        scale_pos_weight=SCALE_POS_WEIGHT,
                        n_jobs=-1,random_state=42,verbose=-1)
    lgb.fit(X_train,y_train,eval_set=[(X_val,y_val)],
            callbacks=[early_stopping(40,verbose=False),log_evaluation(100)])
    vm=evaluate(lgb,X_val,y_val,"val"); tm=evaluate(lgb,X_test,y_test,"test")
    mlflow.log_params({"n_estimators":600,"best_iter":lgb.best_iteration_})
    mlflow.log_metrics({**vm,**tm}); log_cm(lgb,X_test,y_test,"lgb")
    sig=infer_signature(X_train,lgb.predict_proba(X_train))
    mlflow.sklearn.log_model(lgb,"model",signature=sig); LGB_RUN=run.info.run_id
print(f"LGB anomaly  val_auc={vm['val_auc']}  test_auc={tm['test_auc']}")

In [ ]:
# ── CELL 15 : MODEL 3 — Signal voting ensemble (interpretable) ────────────────
# Pure voting ensemble: predicts anomaly if >= SIGNAL_THRESHOLD signals active.
# Effective signals: S1=SPC, S2=IsolationForest, S3=Tap rejection (3 active).
# S4 (network) and S5 (bus_ops) always 0 — do not contribute to vote count.

# ── DEPLOYMENT NOTE ─────────────────────────────────────────────────────────
# SignalVotingEnsemble is defined inline. SageMaker cannot deserialise an
# inline-defined class from joblib at endpoint inference time.
# Before deploying the endpoint, copy this class into a standalone
# inference.py script and package it alongside model.tar.gz.
# Training, evaluation, and MLflow logging below work correctly as-is.
# ─────────────────────────────────────────────────────────────────────────────
class SignalVotingEnsemble:
    """2-of-3 active signal voting anomaly detector (S4 network + S5 bus_ops inactive)."""
    def __init__(self, signal_cols, threshold=2):
        self.signal_cols = signal_cols
        self.threshold   = threshold

    def fit(self, X, y=None):
        return self

    def predict(self, X_df):
        return (X_df[self.signal_cols].sum(axis=1) >= self.threshold).astype(int).values

    def predict_proba(self, X_df):
        active = X_df[self.signal_cols].sum(axis=1).values
        n_active = max(len(self.signal_cols), 1)
        prob_1 = np.clip(active / n_active, 0, 1)
        return np.column_stack([1-prob_1, prob_1])

    def get_params(self, deep=True):
        return {"signal_cols": self.signal_cols, "threshold": self.threshold}

avail_signals = [c for c in SIGNAL_COLS if c in test_df.columns]
if avail_signals:
    voter = SignalVotingEnsemble(avail_signals, SIGNAL_THRESHOLD)
    voter.fit(train_df)

    vm_v = evaluate(voter, val_df,  y_val,  "val")
    tm_v = evaluate(voter, test_df, y_test, "test")

    with mlflow.start_run(run_name="signal_voting_ensemble") as run:
        mlflow.set_tags({**COMMON_TAGS,"model_type":"signal_voting_deterministic"})
        mlflow.log_params({"signals":str(avail_signals),"threshold":SIGNAL_THRESHOLD})
        mlflow.log_metrics({**vm_v,**tm_v})
        joblib.dump(voter,"/tmp/ps4_voter.joblib")
        mlflow.log_artifact("/tmp/ps4_voter.joblib","model")
        VOTER_RUN=run.info.run_id
    print(f"Voting ensemble  val_auc={vm_v['val_auc']}  test_auc={tm_v['test_auc']}")
else:
    print("Signal columns not in test set — voting ensemble skipped"); VOTER_RUN=None

In [ ]:
# ── CELL 16 : Optuna — tune XGBoost (all-fold) ───────────────────────────────
TS_CV=TimeSeriesSplit(n_splits=5)

def optuna_obj(trial):
    p=dict(
        n_estimators    =trial.suggest_int("n_estimators",200,800),
        max_depth       =trial.suggest_int("max_depth",4,9),
        learning_rate   =trial.suggest_float("learning_rate",0.01,0.2,log=True),
        subsample       =trial.suggest_float("subsample",0.5,1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree",0.5,1.0),
        min_child_weight=trial.suggest_int("min_child_weight",1,30),
        gamma           =trial.suggest_float("gamma",0.0,3.0),
        scale_pos_weight=SCALE_POS_WEIGHT,
        tree_method="hist",random_state=42,n_jobs=-1)
    scores=[]
    for step,(tr,va) in enumerate(TS_CV.split(X_cv)):
        m=XGBClassifier(**p)
        m.fit(X_cv[tr],y_cv[tr],eval_set=[(X_cv[va],y_cv[va])],verbose=False)
        scores.append(roc_auc_score(y_cv[va],m.predict_proba(X_cv[va])[:,1]))
        trial.report(np.mean(scores),step)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(scores)

study=optuna.create_study(direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10))
study.optimize(optuna_obj,n_trials=50,show_progress_bar=True)
BEST_PARAMS={**study.best_params,"scale_pos_weight":SCALE_POS_WEIGHT,
             "tree_method":"hist","random_state":42,"n_jobs":-1}

with mlflow.start_run(run_name="xgboost_optuna") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"xgboost_optuna_tuned"})
    mlflow.log_params({**BEST_PARAMS,"optuna_trials":50,"optuna_cv":round(study.best_value,4)})
    xgb_tuned=XGBClassifier(**BEST_PARAMS)
    xgb_tuned.fit(X_cv,y_cv,verbose=False)
    vm=evaluate(xgb_tuned,X_val,y_val,"val"); tm=evaluate(xgb_tuned,X_test,y_test,"test")
    mlflow.log_metrics({**vm,**tm}); log_cm(xgb_tuned,X_test,y_test,"xgb_optuna")
    sig=infer_signature(X_train,xgb_tuned.predict_proba(X_train))
    mlflow.sklearn.log_model(xgb_tuned,"model",signature=sig)
    XGB_TUNED_RUN=run.info.run_id
print(f"XGB(Optuna) val_auc={vm['val_auc']}  test_auc={tm['test_auc']}")

In [ ]:
# ── CELL 17 : Stacking + SHAP ────────────────────────────────────────────────
with mlflow.start_run(run_name="stacking_ensemble") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"stacking_ensemble"})
    base=[
        ("xgb",XGBClassifier(**{**BEST_PARAMS,"n_estimators":300},verbosity=0)),
        ("lgb",LGBMClassifier(n_estimators=300,learning_rate=0.05,
                               scale_pos_weight=SCALE_POS_WEIGHT,
                               n_jobs=-1,random_state=42,verbose=-1)),
        ("rf", RandomForestClassifier(n_estimators=200,max_depth=10,
                                       class_weight="balanced_subsample",
                                       n_jobs=-1,random_state=42)),
    ]
    stack=StackingClassifier(
        estimators=base,
        final_estimator=CalibratedClassifierCV(
            LogisticRegression(C=1.0,max_iter=500),cv=3,method="isotonic"),
        cv=TimeSeriesSplit(n_splits=5),stack_method="predict_proba",
        passthrough=True,n_jobs=-1)
    stack.fit(X_cv,y_cv)
    vm=evaluate(stack,X_val,y_val,"val"); tm=evaluate(stack,X_test,y_test,"test")
    mlflow.log_metrics({**vm,**tm}); log_cm(stack,X_test,y_test,"stack")
    sig=infer_signature(X_train,stack.predict_proba(X_train))
    mlflow.sklearn.log_model(stack,"model",signature=sig); STACK_RUN=run.info.run_id
print(f"Stack  val_auc={vm['val_auc']}  test_auc={tm['test_auc']}")

# SHAP
X_shap=pd.DataFrame(X_test[:1500],columns=FEATURE_COLS)
explainer=shap.TreeExplainer(xgb_tuned); sv=explainer(X_shap)
with mlflow.start_run(run_name="shap_analysis") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"shap_analysis"})
    shap.plots.bar(sv,max_display=20,show=False)
    fig = plt.gcf(); ax = plt.gca()
    ax.set_title("PS4 XGBoost — SHAP Feature Importance",pad=12)
    fig.tight_layout(); fig.savefig("/tmp/ps4_shap.png",dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/ps4_shap.png")

In [ ]:
# ── CELL 18 : Select champion + register (MLflow + SM) ───────────────────────
all_runs=mlflow.search_runs(experiment_names=[EXPERIMENT_NAME],
                             order_by=["metrics.test_auc DESC"])
all_runs=all_runs[all_runs["tags.mlflow.runName"].str.contains(
    "xgboost|lightgbm|stack",na=False)]
CHAMPION_RUN_ID=all_runs.iloc[0]["run_id"]
CHAMPION_NAME  =all_runs.iloc[0]["tags.mlflow.runName"]
CHAMPION_AUC   =all_runs.iloc[0]["metrics.test_auc"]
print(f"Champion: {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}")

mlfc=mlflow.tracking.MlflowClient()
try: mlfc.create_registered_model(MLFLOW_MODEL_NAME,
        description="PS4 Hourly Anomaly Detection — Chicago Ventra")
except: pass
mv=mlfc.create_model_version(name=MLFLOW_MODEL_NAME,
    source=f"runs:/{CHAMPION_RUN_ID}/model",run_id=CHAMPION_RUN_ID,
    tags={"algorithm":CHAMPION_NAME,"test_auc":str(CHAMPION_AUC),
          "gold_version":str(GOLD_VERSION),"hdbscan_run":str(HDBSCAN_RUN)})
mlfc.transition_model_version_stage(MLFLOW_MODEL_NAME,mv.version,"Staging")
MLFLOW_VERSION=mv.version
print(f"MLflow: {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION} → Staging")

champion_model=mlflow.sklearn.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
joblib.dump(champion_model,"/tmp/ps4_champion.joblib")
joblib.dump(FEATURE_COLS,"/tmp/ps4_feature_cols.joblib")
test_metrics=evaluate(champion_model,X_test,y_test,"test")
S3_CLIENT.put_object(Bucket=ARTIFACT_BUCKET,Key="sagemaker/ps4/metrics/test_metrics.json",
                     Body=json.dumps(test_metrics),ContentType="application/json")
with tarfile.open("/tmp/model.tar.gz","w:gz") as tar:
    tar.add("/tmp/ps4_champion.joblib",arcname="ps4_champion.joblib")
    tar.add("/tmp/ps4_feature_cols.joblib",arcname="ps4_feature_cols.joblib")
MODEL_S3_KEY=f"sagemaker/ps4/model-v{MLFLOW_VERSION}/model.tar.gz"
S3_CLIENT.upload_file("/tmp/model.tar.gz",ARTIFACT_BUCKET,MODEL_S3_KEY)

import sagemaker.image_uris as sm_images
image_uri=sm_images.retrieve(
    framework="sklearn", region=REGION, version="1.2-1",
    instance_type="ml.m5.large", image_scope="inference")
try: SM_CLIENT.create_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageGroupDescription="PS4 Anomaly Detection — Chicago Ventra")
except: pass
resp=SM_CLIENT.create_model_package(
    ModelPackageGroupName=MODEL_REGISTRY_NAME,
    ModelPackageDescription=f"{CHAMPION_NAME}|AUC={CHAMPION_AUC:.4f}|Gv{GOLD_VERSION}",
    InferenceSpecification={
        "Containers":[{"Image":image_uri,"ModelDataUrl":f"s3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}"}],
        "SupportedContentTypes":["text/csv"],"SupportedResponseMIMETypes":["application/json"],
        "SupportedRealtimeInferenceInstanceTypes":["ml.m5.large"]},
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={"ps":"PS4","algorithm":CHAMPION_NAME,
        "test_auc":str(round(CHAMPION_AUC,4)),"mlflow_version":str(MLFLOW_VERSION),
        "hdbscan_clusters":str(n_clusters if "n_clusters" in dir() else "N/A")})
SM_MODEL_PKG_ARN=resp["ModelPackageArn"]
print(f"SM Package: {SM_MODEL_PKG_ARN}")

In [ ]:
# ── CELL 19 : Deploy + Monitors (hourly schedule — PS4 is hourly grain) ───────
from sagemaker.model import ModelPackage
from sagemaker.model_monitor import (DefaultModelMonitor,DataCaptureConfig,
                                      CronExpressionGenerator)
from sagemaker.model_monitor.dataset_format import DatasetFormat

SM_CLIENT.update_model_package(ModelPackageArn=SM_MODEL_PKG_ARN,
                                ModelApprovalStatus="Approved")
registered=ModelPackage(role=ROLE,model_package_arn=SM_MODEL_PKG_ARN,
                         sagemaker_session=SM_SESSION)
predictor=registered.deploy(
    initial_instance_count=1,instance_type="ml.m5.large",
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=DataCaptureConfig(
        enable_capture=True,sampling_percentage=100,
        destination_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps4/data-capture/",
        capture_options=["Input","Output"]))
print(f"Endpoint: {ENDPOINT_NAME}")

monitor=DefaultModelMonitor(role=ROLE,instance_count=1,
    instance_type="ml.m5.large",sagemaker_session=SM_SESSION)
pd.DataFrame(X_train,columns=FEATURE_COLS).to_csv("/tmp/ps4_baseline.csv",index=False)
S3_CLIENT.upload_file("/tmp/ps4_baseline.csv",ARTIFACT_BUCKET,
                      "sagemaker/ps4/baseline/baseline.csv")
monitor.suggest_baseline(
    baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps4/baseline/baseline.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps4/baseline/output/",
    wait=True,logs=False)
# PS4 is hourly grain — monitor every 6 hours (matches Evidently AI drift cycle)
monitor.create_monitoring_schedule(
    monitor_schedule_name="chicago-ps4-data-quality",endpoint_input=ENDPOINT_NAME,
    output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps4/monitor-output/",
    statistics=monitor.baseline_statistics(),constraints=monitor.suggested_constraints(),
    schedule_cron_expression="cron(0 */6 * * ? *)",   # every 6h — hourly grain
    enable_cloudwatch_metrics=True)
print("PS4 monitor: every 6 hours (hourly grain)")

cw=boto3.client("cloudwatch",region_name=REGION)
cw.put_metric_alarm(AlarmName="PS4-DataQuality-ViolationRate",
    ComparisonOperator="GreaterThanThreshold",EvaluationPeriods=1,
    MetricName="feature_baseline_drift_percentage",
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    Period=21600,Statistic="Average",Threshold=3.0,
    AlarmDescription="PS4 anomaly input violation >3% — CUBIC MARS 97% gate",
    Dimensions=[{"Name":"Endpoint","Value":ENDPOINT_NAME}],
    TreatMissingData="notBreaching")
print("Monitor + alarm configured.")

In [ ]:
# ── CELL 20 : Summary ─────────────────────────────────────────────────────────
print("="*60)
print("PS4 — Hourly Anomaly Detection — Complete")
print("="*60)
print(f"Champion     : {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}")
print(f"MLflow model : {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION}")
print(f"SM endpoint  : {ENDPOINT_NAME}")
print(f"Monitor freq : every 6h (hourly grain — Evidently KS+PSI aligned)")
print(f"\nSignal breakdown:")
for s in SIGNAL_COLS:
    rate=(df[s].mean() if s in df.columns else 0)
    print(f"  {s}: {rate:.2%} active")
print(f"\nHDBSCAN: {n_clusters if 'n_clusters' in dir() else 'N/A'} clusters")
print(f"\nFeasibility gap: 5%")
